In [1]:
# 1. Load cleaned data
import pandas as pd
import numpy as np
import joblib
import os

# load cleaned data (after imputation)
X_train = pd.read_csv("/content/X_train_clean.csv")
y_train = pd.read_csv("/content/y_train.csv").values.ravel()

print("Shape:", X_train.shape)

Shape: (4504, 18)


In [2]:
# 2. Identify categorical columns
# detect categorical columns (object or category)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:", cat_cols)

Categorical columns: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']


In [3]:
# 3. LIGHTGBM BRANCH
# 3.1. Label Encoding
from sklearn.preprocessing import LabelEncoder

# create dictionary to store encoders
lgb_encoders = {}

# copy data to avoid modifying original
X_train_lgb = X_train.copy()

for col in cat_cols:
    le = LabelEncoder()

    # convert to string (safe for encoding)
    X_train_lgb[col] = X_train_lgb[col].astype(str)

    # fit encoder ONLY on train (no leakage)
    X_train_lgb[col] = le.fit_transform(X_train_lgb[col])

    # save encoder
    lgb_encoders[col] = le

print("Label Encoding DONE")

Label Encoding DONE


In [4]:
# 3.2. Save encoder
joblib.dump(lgb_encoders, "/content/lgb_encoder.pkl")

print("Saved lgb_encoder.pkl")

Saved lgb_encoder.pkl


In [5]:
# 3.3. Compute imbalance weight
# count class distribution
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)

# compute scale_pos_weight
scale_pos_weight = n_neg / n_pos

print("n_pos:", n_pos)
print("n_neg:", n_neg)
print("scale_pos_weight:", scale_pos_weight)

n_pos: 758
n_neg: 3746
scale_pos_weight: 4.941952506596306


In [6]:
# 3.4. Save LGB data
np.savez(
    "/content/lgb_train_data.npz",
    X_train=X_train_lgb,
    y_train=y_train,
    scale_pos_weight=scale_pos_weight
)

print("Saved lgb_train_data.npz")

Saved lgb_train_data.npz


In [7]:
# 4. CATBOOST BRANCH
# 4.1. Keep raw categorical
# copy data (NO encoding)
X_train_cb = X_train.copy()

In [8]:
# 4.2. Get categorical indices
# CatBoost needs column index, not name
cat_features = [X_train_cb.columns.get_loc(col) for col in cat_cols]

print("cat_features index:", cat_features)

cat_features index: [13, 14, 15, 16, 17]


In [9]:
# 4.3. Compute class weights
# same logic but different format
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)

# class_weights format: [weight_for_0, weight_for_1]
class_weights = [1, n_neg / n_pos]

print("class_weights:", class_weights)

class_weights: [1, np.float64(4.941952506596306)]


In [10]:
# 4.4. Save CatBoost data
np.savez(
    "/content/cb_train_data.npz",
    X_train=X_train_cb,
    y_train=y_train,
    cat_features=cat_features,
    class_weights=class_weights
)

print("Saved cb_train_data.npz")

Saved cb_train_data.npz
